# Text Style Transfer

## HuggingFace Login

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Pre-processing

### Import Dataset

In [ ]:
# Dataset repo
!git clone https://github.com/lijuncen/Sentiment-and-Style-Transfer.git

### Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --quiet
!pip install -q -U transformers accelerate
!pip install sentencepiece sacremoses --quiet
!pip install nltk scikit-learn --quiet
!pip install evaluate --quiet
!pip install -q --upgrade transformers
!pip install wandb --quiet
!pip install sacrebleu --quiet
!pip install nlpaug sentencepiece datasets --quiet

### Prepare Dataset

In [ ]:
import os, random, pickle
import pandas as pd

random.seed(53)

ref0_path = "/content/Sentiment-and-Style-Transfer/data/yelp/reference.0"  # neg\tpos
ref1_path = "/content/Sentiment-and-Style-Transfer/data/yelp/reference.1"  # pos\tneg

def read_tsv_lines(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]
    return lines

lines0 = read_tsv_lines(ref0_path)
lines1 = read_tsv_lines(ref1_path)

pairs = []

# reference.1: left=pos, right=neg
for ln in lines1:
    if '\t' not in ln:
        continue
    left, right = ln.split('\t', 1)
    pos = left.strip()
    neg = right.strip()
    pairs.append({'pos': pos, 'neg': neg})

# reference.0: left=neg, right=pos -> swap to pos,neg
for ln in lines0:
    if '\t' not in ln:
        continue
    left, right = ln.split('\t', 1)
    neg = left.strip()
    pos = right.strip()
    pairs.append({'pos': pos, 'neg': neg})

print("Total pairs combined:", len(pairs))

# paper split: 400 train, 100 dev, 500 test
test_pairs  = pairs[500:1000]
pairs = pairs[:500]
random.shuffle(pairs)
train_pairs = pairs[:400]
dev_pairs   = pairs[400:500]

# convert to DataFrame
train_df = pd.DataFrame(train_pairs)
dev_df   = pd.DataFrame(dev_pairs)
test_df  = pd.DataFrame(test_pairs)

# save datasets
os.makedirs("/content/text-style-transfer/data", exist_ok=True)
train_df.to_csv("/content/text-style-transfer/data/train.csv", index=False)
dev_df.to_csv("/content/text-style-transfer/data/dev.csv", index=False)
test_df.to_pickle("/content/text-style-transfer/data/test.pkl")


print("Wrote data/train.csv, data/dev.csv, data/test.pkl")
print("Train size:", len(train_df), "Dev size:", len(dev_df), "Test size:", len(test_df))


# unparallel Data
unparallel_ref1_path = "/content/Sentiment-and-Style-Transfer/data/yelp/sentiment.dev.1"  #
lines = []
with open(unparallel_ref1_path, 'r', encoding='utf-8') as f:
    lines = [ln.strip() for ln in f if ln.strip()]
print("Total unparallel lines:", len(lines))

unparallel_data_df = pd.DataFrame({'pos': lines})
unparallel_data_df.to_pickle("/content/text-style-transfer/data/unparallel.pkl")
print("Wrote data/unparallel.pkl")

## Data Augmentation

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')


In [ ]:
import os
import random
import pandas as pd
import nlpaug.augmenter.word as naw
from tqdm import tqdm

random.seed(53)

# --- Load data ---
train_path = "/content/text-style-transfer/data/train.csv"
dev_path   = "/content/text-style-transfer/data/dev.csv"

train_df = pd.read_csv(train_path)
dev_df   = pd.read_csv(dev_path)

# --- create augmenters ---
augmenters = {
    "spelling": naw.SpellingAug(),
    "bert": naw.ContextualWordEmbsAug(model_path='bert-base-uncased', action='substitute'),
    "synonym": naw.SynonymAug(aug_src='wordnet'),
    "swap": naw.RandomWordAug(action="swap"),
    "delete": naw.RandomWordAug(action="delete")
}

# --- custom split function (replacement for SplitAug) ---
def split_random_word(text, prob=0.1):
    words = text.split()
    new_words = []
    for w in words:
        if len(w) > 4 and random.random() < prob:
            cut = random.randint(1, len(w)-1)
            new_words.extend([w[:cut], w[cut:]])
        else:
            new_words.append(w)
    return " ".join(new_words)

# --- apply random augmentation with 50% chance ---
def augment_df(df, prob=0.5):
    augmented_rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        if random.random() < prob:
            aug_type = random.choice(list(augmenters.keys()) + ["split"])
            # Only augment the positive (non-toxic) text
            if aug_type == "split":
                new_pos = split_random_word(row["pos"])
            else:
                aug = augmenters[aug_type]
                new_pos = aug.augment(row["pos"])
            # Keep the negative (toxic) text unchanged
            new_neg = row["neg"]
            augmented_rows.append({
                "pos": new_pos,
                "neg": new_neg,
                "aug_type": aug_type
            })
    return pd.DataFrame(augmented_rows)

# --- run augmentation ---
aug_train_df = augment_df(train_df)
aug_dev_df   = augment_df(dev_df)

print(f"\nOriginal train: {len(train_df)}, Augmented train: {len(aug_train_df)}")
print(f"Original dev: {len(dev_df)}, Augmented dev: {len(aug_dev_df)}")

# --- combine and save ---
train_all = pd.concat([train_df, aug_train_df[["pos", "neg"]]], ignore_index=True)
dev_all   = pd.concat([dev_df, aug_dev_df[["pos", "neg"]]], ignore_index=True)

os.makedirs("/content/text-style-transfer/data", exist_ok=True)
train_all.to_csv("/content/text-style-transfer/data/train.csv", index=False)
dev_all.to_csv("/content/text-style-transfer/data/dev.csv", index=False)

print("✅ Augmented datasets (only pos) saved in /content/text-style-transfer/data")


## Train BERT Style Classifier


### Train BERT as Style Classifier on YELP Reference Data

In [ ]:
import os
import random
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# load full Yelp reference parallel data

random.seed(53)

ref0_path = "/content/Sentiment-and-Style-Transfer/data/yelp/reference.0"  # neg\tpos
ref1_path = "/content/Sentiment-and-Style-Transfer/data/yelp/reference.1"  # pos\tneg

def read_tsv_lines(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]
    return lines

lines0 = read_tsv_lines(ref0_path)
lines1 = read_tsv_lines(ref1_path)

pairs = []

# reference.1: left=pos, right=neg
for ln in lines1:
    if '\t' not in ln:
        continue
    left, right = ln.split('\t', 1)
    pos = left.strip()
    neg = right.strip()
    pairs.append({'pos': pos, 'neg': neg})

# reference.0: left=neg, right=pos
for ln in lines0:
    if '\t' not in ln:
        continue
    left, right = ln.split('\t', 1)
    neg = left.strip()
    pos = right.strip()
    pairs.append({'pos': pos, 'neg': neg})

print(f"✅ Total pairs loaded: {len(pairs)} (should be 1000)")

# flatten to text + label lists
pos_texts = [p['pos'] for p in pairs]
neg_texts = [p['neg'] for p in pairs]

texts = pos_texts + neg_texts
labels = [1] * len(pos_texts) + [0] * len(neg_texts)  # 1=POS, 0=NEG

print(f"Total labeled samples: {len(texts)} (Pos={len(pos_texts)}, Neg={len(neg_texts)})")

# split into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=53, stratify=labels
)

print(f"Train size: {len(train_texts)}, Val size: {len(val_texts)}")


# tokenization
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

from torch.utils.data import Dataset

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
val_enc = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

train_dataset = SentimentDataset(train_enc, train_labels)
val_dataset = SentimentDataset(val_enc, val_labels)


# load BERT model and Trainer setup
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./bert_style_classifier",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_dir="./bert_style_classifier/logs",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# train the model
trainer.train()

print(f"\n✅ BERT style classifier trained and saved to {save_dir}")


### Push Model to Hub

In [ ]:
from huggingface_hub import HfApi, create_repo
from transformers import BertForSequenceClassification, BertTokenizer

repo_name = "bert-text-style-classifier-yelp"
username = "shubhamkr"

# create the repo on Hugging Face (if not exists)
create_repo(f"{username}/{repo_name}", exist_ok=True)

# save locally first
save_dir = f"./{repo_name}"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# then push to Hub
model.push_to_hub(f"{username}/{repo_name}")
tokenizer.push_to_hub(f"{username}/{repo_name}")


### Test Style Classifier Accuracy

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "shubhamkr/bert-text-style-classifier-yelp"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()


In [36]:
test_df = pd.read_pickle("/content/text-style-transfer/data/test.pkl")

pos_texts = test_df["pos"].tolist()
neg_texts = test_df["neg"].tolist()

# Combine into one list of texts + labels
texts = pos_texts + neg_texts
labels = [1]*len(pos_texts) + [0]*len(neg_texts)

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 16
correct = 0
all_preds = []
all_labels = []

for i in tqdm(range(0, len(texts), batch_size), desc="Evaluating"):
    batch_texts = texts[i:i+batch_size]
    batch_labels = labels[i:i+batch_size]

    enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(batch_labels)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(all_labels, all_preds)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%\n")

print("Detailed report:")
print(classification_report(all_labels, all_preds, target_names=["NEGATIVE", "POSITIVE"]))


## Evaluation Code

In [7]:
import torch
import sys
import random
import numpy as np
import pickle
import argparse
import evaluate
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline, GPT2Tokenizer, GPT2LMHeadModel, set_seed

class AutomaticEvaluator:
    def __init__(self, seed_value):
        self.seed_value = seed_value
        self.bleu = None
        self.sim_model = None
        self.critic_model = None
        self.sentiment_analysis = None
        self.gpt2_tokenizer = None
        self.gpt2_model = None

    def set_seed(self):
        random.seed(self.seed_value)
        np.random.seed(self.seed_value)
        torch.manual_seed(self.seed_value)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.seed_value)
            torch.cuda.manual_seed_all(self.seed_value)
        set_seed(self.seed_value)

    def load_models(self):
        print("Loading evaluation models...")
        self.bleu = evaluate.load("bleu")
        self.sim_model = SentenceTransformer('bert-base-nli-mean-tokens')
        self.critic_model = pipeline("text-classification", model="roberta-large-mnli")
        self.sentiment_analysis = pipeline("sentiment-analysis", model='distilbert-base-uncased-finetuned-sst-2-english')

        with torch.no_grad():
            self.gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
            self.gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2')
            self.gpt2_model.eval()
        print("Models loaded successfully.")

    def gpt2_score(self, sentence):
        tokenize_input = self.gpt2_tokenizer.encode(sentence, return_tensors='pt')
        with torch.no_grad():
            loss = self.gpt2_model(tokenize_input, labels=tokenize_input).loss
        return (torch.exp(loss).item())*0.5

    def similarity(self, sentence1, sentence2):
        embeddings = self.sim_model.encode([sentence1, sentence2])
        sim_score = cosine_similarity([embeddings[0]], [embeddings[1]])
        return sim_score[0][0]

    def senti_score(self, sentence):
        return self.sentiment_analysis(sentence)[0]

    def evaluate_all(self, srcs_file, refs_file, preds_file, target_label):
        with open(srcs_file, 'rb') as f:
            sources = pickle.load(f)
        with open(refs_file, 'rb') as f:
            references = pickle.load(f)
        with open(preds_file, 'rb') as f:
            predictions = pickle.load(f)

        print(f"Evaluating {len(predictions)} predictions...")

        references_bleu = [[ref] for ref in references]
        sources_bleu = [[src] for src in sources]

        sim_scores = []
        gpt2_scores = []
        correct_count = 0

        for i in range(len(predictions)):
            pred = predictions[i]
            ref = references[i]

            if self.senti_score(pred)['label'] == target_label:
                correct_count += 1

            sim_scores.append(self.similarity(pred, ref))

            gpt2_scores.append(self.gpt2_score(pred))

        bleu_withsrc = self.bleu.compute(predictions=predictions, references=sources_bleu, max_order=4)
        bleu_withtrg = self.bleu.compute(predictions=predictions, references=references_bleu, max_order=4)

        acc = correct_count / len(predictions)
        avg_sim = sum(sim_scores) / len(sim_scores)
        avg_gpt2_ppl = sum(gpt2_scores) / len(gpt2_scores)

        return (
            acc,
            bleu_withsrc['bleu'],
            bleu_withtrg['bleu'],
            avg_sim,
            avg_gpt2_ppl
        )


def main(args):
    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()

    target_label = 'NEGATIVE' if args.task == 'pos_to_neg' else 'POSITIVE'

    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
        target_label=target_label
    )

    print('-' * 20)
    print(f"Evaluation Results for Task: {args.task}")
    print('-' * 20)
    print(f"Accuracy: {acc:.4f}")
    print(f"Bleu with Source: {bleu_withsrc:.4f}")
    print(f"Bleu with Target: {bleu_withtrg:.4f}")
    print(f"Similarity: {sim:.4f}")
    print(f"GPT-2 PPL (Fluency): {gpt2_ppl:.4f}")


## Train BART for TST

### Load Style Classifier

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

style_model_name = "shubhamkr/bert-text-style-classifier-yelp"
style_tokenizer = AutoTokenizer.from_pretrained(style_model_name)
style_classifier = AutoModelForSequenceClassification.from_pretrained(style_model_name).to(device)
style_classifier.eval()


### Create Style Reward Trainer

In [9]:
from transformers import Seq2SeqTrainer
import torch.nn.functional as F
import torch


class StyleRewardTrainer(Seq2SeqTrainer):
    def __init__(self, style_classifier, style_tokenizer, alpha=0.5, target_label=0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.style_classifier = style_classifier
        self.style_tokenizer = style_tokenizer
        self.alpha = alpha
        self.target_label = target_label  # 0=NEGATIVE, 1=POSITIVE

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
      outputs = model(**inputs)
      ce_loss = outputs.loss

      processor = getattr(self, "processing_class", self.tokenizer)

      # generate predictions
      generated_ids = model.generate(
          input_ids=inputs["input_ids"],
          attention_mask=inputs["attention_mask"],
          max_length=50
      )
      preds = processor.batch_decode(generated_ids, skip_special_tokens=True)

      if not preds or all(len(p.strip()) == 0 for p in preds):
          return (ce_loss, outputs) if return_outputs else ce_loss

      # compute style reward
      style_inputs = self.style_tokenizer(preds, padding=True, truncation=True, return_tensors="pt").to(model.device)
      with torch.no_grad():
          logits = self.style_classifier(**style_inputs).logits
          probs = torch.softmax(logits, dim=-1)
          target_probs = probs[:, self.target_label]
          rewards = 2 * target_probs - 1  # map 0–1 → −1..+1

      # safe normalization
      std = rewards.std()
      if torch.isnan(std) or std < 1e-6:
          norm_rewards = rewards - rewards.mean()
      else:
          norm_rewards = (rewards - rewards.mean()) / (std + 1e-8)

      # weighted loss (detach reward term)
      total_loss = (1 - self.alpha) * ce_loss + self.alpha * norm_rewards.mean().detach()

      return (total_loss, outputs) if return_outputs else total_loss


### Train Model Followed by Self Training

In [12]:
import shutil
import os
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import argparse
import torch
import math
import sacrebleu
from tqdm import tqdm
from scipy.stats.mstats import gmean
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    BartConfig
)
from huggingface_hub import HfApi, create_repo

def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

def preprocess_data(src_data, trg_data):
    df = pd.DataFrame({src_col: src_data[src_col], trg_col: trg_data[trg_col]})
    df = df.sample(frac=1)
    return df

def tokenize_datasets(df, src_col, trg_col, tokenizer, max_length):
    src_encodings = tokenizer(
        df[src_col].values.tolist(),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    trg_encodings = tokenizer(
        df[trg_col].values.tolist(),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    dataset = CreateDataset(src_encodings, trg_encodings)
    return dataset

def train_model(model, train_dataset, dev_dataset, tokenizer, args):
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    trainer = StyleRewardTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        data_collator=data_collator,
        tokenizer=tokenizer,
        style_classifier=style_classifier,
        style_tokenizer=style_tokenizer,
        alpha=0.5,
        target_label=0
    )
    trainer.processing_class = tokenizer
    trainer.tokenizer = tokenizer

    trainer.train()
    return model

def generate_predictions(model, test_data, src_col, tokenizer, output_file):
    predictions = []
    for idx in range(len(test_data[src_col])):
        src = test_data[src_col].values[idx]
        src_tknz = tokenizer(src, truncation=True, padding=True, max_length=args.max_length, return_tensors='pt')
        generated_ids = model.generate(src_tknz["input_ids"].cuda(), max_length=args.max_length)
        prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(prediction)

    with open(output_file, 'wb') as f:
        pickle.dump(remove_prefix(predictions, 'NEG: '), f)

def generate_synthetic_data(model, unparallel_data, src_col, tokenizer):
    print(f"🔄 Generating synthetic train data from unparallel data...")
    predictions = []
    for idx in range(len(unparallel_data[src_col])):
        src = unparallel_data[src_col].values[idx]
        src = "POS: " + src;
        src_tknz = tokenizer(src, truncation=True, padding=True, max_length=args.max_length, return_tensors='pt')
        generated_ids = model.generate(src_tknz["input_ids"].cuda(), max_length=args.max_length)
        prediction = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(prediction)

    data_pairs = pd.DataFrame({
            'pos': unparallel_data[src_col].values.tolist(),
            'neg': remove_prefix(predictions, 'NEG: ')
    })
    return data_pairs

def evaluate_transfer(critic_model, src, pred, target_label, threshold=0.9):
    res = critic_model(f"{src} </s> {pred}")[0]
    label = res['label']
    score = res['score']

    # Flip detection based on NLI label and target
    if target_label == "NEGATIVE" and label == "CONTRADICTION" and score >= threshold:
        style_prob = 1.0
    elif target_label == "POSITIVE" and label == "ENTAILMENT" and score >= threshold:
        style_prob = 1.0
    else:
        style_prob = 0.0

    return style_prob, label, score


def compute_scores_for_synthetic_pairs(sources, preds, evaluator, target_label="NEGATIVE", threshold=0.9):
    print(f"🔄 Computing scores for synthetic pairs using NLI evaluator...")
    sim_model = evaluator.sim_model
    critic_model = evaluator.critic_model
    out = []

    for src, pred in tqdm(zip(sources, preds), total=len(preds), desc="Scoring pairs"):
        style_prob, nli_label, nli_score = evaluate_transfer(critic_model, src, pred, target_label, threshold)

        bleu_vs_src = sacrebleu.sentence_bleu(pred, [src]).score / 100.0

        emb = sim_model.encode([src, pred])
        cos_sim = float(np.dot(emb[0], emb[1]) / (np.linalg.norm(emb[0]) * np.linalg.norm(emb[1]) + 1e-9))
        sim_norm = (cos_sim + 1.0) / 2.0

        out.append({
            'src': src,
            'pred': pred,
            'style_prob': float(style_prob),
            'bleu': float(bleu_vs_src),
            'sim': float(sim_norm),
            'nli_label': nli_label,
            'nli_score': float(nli_score)
        })

    print(f"✅ Computed scores for {len(out)} synthetic pairs.")
    return out


def compute_filtered_score(item):
    style_prob = item['style_prob']
    bleu = item['bleu']
    sim = item['sim']

    total_score = style_prob * sim
    return float(total_score)


def add_scores_and_select(scored_list, top_k, high_thresh=0.9, low_thresh=0.7):
    # compute scores for all examples
    for it in scored_list:
        it['score'] = compute_filtered_score(it)

    # sort descending by score
    scored_list.sort(key=lambda x: x['score'], reverse=True)
    # try selecting items above high threshold
    high_selected = [x for x in scored_list if x['score'] >= 0.5 and x['score'] <= 0.75]

    if len(high_selected) >= top_k:
        return high_selected[:top_k]

    # not enough high-score examples → relax threshold
    low_selected = [x for x in scored_list if x['score'] >= low_thresh]

    # return whichever is smaller (to avoid overshooting top_k)
    return low_selected[:min(len(low_selected), top_k)]


def progressive_self_training_round(
        model,
        tokenizer,
        unparallel_path,
        train_csv,
        dev_csv,
        output_dir,
        args,
        target_label,
        top_k,
        round_id,
        num_orig,
        seed_value=53,
        ):

    set_seed(seed_value)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n===== SELF-TRAINING ROUND {round_id} =====")

    # ---------- Stage 1: Generate ----------
    unparallel_df = remove_newline(pd.read_pickle(unparallel_path))
    unparallel_df = unparallel_df.sample(frac = 1, random_state = args.seed_value)
    unparallel_pred_df = generate_synthetic_data(model, unparallel_df, 'pos', tokenizer)
    print("S2 complete!")

    # ---------- Stage 2: Score ----------
    evaluator = AutomaticEvaluator(seed_value)
    evaluator.set_seed()
    evaluator.load_models()
    sources = unparallel_pred_df["pos"].tolist()
    preds = unparallel_pred_df["neg"].tolist()
    scored = compute_scores_for_synthetic_pairs(sources, preds, evaluator, target_label="NEGATIVE")
    print("S3 complete!")

    # ---------- Stage 3: Select ----------
    selected = add_scores_and_select(scored, top_k=top_k)
    print(f"Selected top {len(selected)} pseudo pairs out of {len(scored)}")
    print("S4 complete!")

    # ---------- Stage 4: Merge -----------
    train_df = pd.read_csv(train_csv)
    total_synthetic_cap = 2000
    total_original_train = 596
    sampled_train_df = train_df.sample(n=num_orig, random_state=seed_value)
    print(f"Selected {num_orig} original samples out of {total_original_train} for round {round_id}")
    print("S5 complete!")

    # build pseudo DataFrame
    pseudo_df = pd.DataFrame({
      "neg": [x["pred"] for x in selected],
      "pos": [x["src"] for x in selected]
    })

    # combine sampled original + synthetic
    augmented_train_df = pd.concat([sampled_train_df, pseudo_df], ignore_index=True)
    print(f"Augmented train data size: {len(augmented_train_df)}")


    # ----------- Stage 6: Fine-tune -----------
    print('Finetuning with the augmented data...')
    neg_prompt = 'NEG: '
    pos_prompt = 'POS: '
    augmented_train_df["pos"] = augmented_train_df["pos"].apply(lambda x: pos_prompt+x)
    augmented_train_df["neg"] = augmented_train_df["neg"].apply(lambda x: neg_prompt+x)
    dev_df = remove_newline(pd.read_csv(dev_csv))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)
    dev_df["pos"] = dev_df["pos"].apply(lambda x: pos_prompt+x)
    dev_df["neg"] = dev_df["neg"].apply(lambda x: neg_prompt+x)
    augmented_train_dataset = tokenize_datasets(augmented_train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    training_args = Seq2SeqTrainingArguments(
      output_dir=output_dir,
      eval_strategy="epoch",
      learning_rate = 5e-5 if round_id == 4 else 1e-5,
      per_device_train_batch_size=args.batch_size,
      per_device_eval_batch_size=args.batch_size,
      weight_decay=args.weight_decay,
      save_total_limit=1,
      save_strategy='epoch',
      load_best_model_at_end=True,
      num_train_epochs = 4 if round_id == 4 else 2,
      predict_with_generate=True,
      fp16=True
    )

    # Train the model
    model = train_model(model, augmented_train_dataset, dev_dataset, tokenizer, training_args)
    print("S6 complete!")
    return model


class CreateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])
        return item

    def __len__(self):
        return len(self.labels['input_ids'])

def write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, filepath):
    with open(filepath, 'w') as file:
        file.write('Accuracy: ' + str(acc) + '\n')
        file.write('Bleu with Source: ' + str(bleu_withsrc) + '\n')
        file.write('Bleu with Target: ' + str(bleu_withtrg) + '\n')
        file.write('Similarity: ' + str(sim) + '\n')
        file.write('GPT2 PPL: ' + str(gpt2_ppl) + '\n')

    print('\n\n\n---------RESULTS---------\n')
    print('Accuracy: ' + str(acc) + '\n')
    print('Bleu with Source: ' + str(bleu_withsrc) + '\n')
    print('Similarity: ' + str(sim) + '\n')
    print('GPT2 PPL: ' + str(gpt2_ppl) + '\n')


def remove_prefix(strings, prefix):
    return [string.replace(prefix, '') for string in strings]

def remove_newline(df):
    df = df.apply(lambda x: x.str.replace('\n', ''))
    return df

def main(args):
    shutil.rmtree('facebook', ignore_errors=True)
    set_seed(args.seed_value)

    train_df = remove_newline(pd.read_csv(args.train_file))
    train_df = train_df.sample(frac = 1, random_state = args.seed_value)

    dev_df = remove_newline(pd.read_csv(args.dev_file))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)

    test_df = remove_newline(pd.read_pickle(args.test_file))

    neg_prompt = 'NEG: '
    pos_prompt = 'POS: '
    if args.prompt_enabled:

        train_df["pos"] = train_df["pos"].apply(lambda x: pos_prompt+x)
        train_df["neg"] = train_df["neg"].apply(lambda x: neg_prompt+x)

        dev_df["pos"] = dev_df["pos"].apply(lambda x: pos_prompt+x)
        dev_df["neg"] = dev_df["neg"].apply(lambda x: neg_prompt+x)

        test_df["pos"] = test_df["pos"].apply(lambda x: pos_prompt+x)
        test_df["neg"] = test_df["neg"].apply(lambda x: neg_prompt+x)

    with open('./output/pos_to_neg/src.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.src_col].values.tolist(), pos_prompt), f)

    with open('./output/pos_to_neg/trg.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.trg_col].values.tolist(), neg_prompt), f)

    tokenizer = BartTokenizer.from_pretrained(args.model_name)
    train_dataset = tokenize_datasets(train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)

    # Load the BART model
    model = BartForConditionalGeneration.from_pretrained(args.model_name)

    config = BartConfig.from_pretrained(args.model_name)
    config.dropout = 0.15
    config.attention_dropout = 0.05
    config.activation_dropout = 0.05
    config.label_smoothing_factor = 0.05

    model.config = config

    # Define a directory to save the model
    output_directory = "./results"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Initialize training arguments correctly
    training_args = Seq2SeqTrainingArguments(
      output_dir=output_directory,
      eval_strategy="epoch",
      learning_rate=args.learning_rate,
      per_device_train_batch_size=args.batch_size,
      per_device_eval_batch_size=args.batch_size,
      weight_decay=args.weight_decay,
      save_total_limit=1,
      save_strategy='epoch',
      load_best_model_at_end=True,
      num_train_epochs= 3,
      predict_with_generate=True,
      fp16=True
    )

    # Train the model
    model = train_model(model, train_dataset, dev_dataset, tokenizer, training_args)

    top_k = 500
    maxtopk = 2000
    num_orig = 196

    for round_id in range(1, 5):
      model = progressive_self_training_round(
            model=model,
            tokenizer=tokenizer,
            unparallel_path=args.unparallel_data,
            train_csv=args.train_file,
            dev_csv=args.dev_file,
            output_dir='./results/selftrain',
            args = args,
            target_label='NEGATIVE',
            top_k=top_k,
            round_id=round_id,
            num_orig=num_orig
        )
      top_k = min(top_k*2, maxtopk)
      num_orig += 100
      print(f"✅ Round {round_id} completed.")


    # Generate predictions on test data
    generate_predictions(model, test_df, args.src_col, tokenizer, args.pred_file) ###
    print('Completed.')

    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()

    target_label = 'NEGATIVE' if args.task == 'pos_to_neg' else 'POSITIVE'

    # <-- Call evaluate_all with the new arguments from argparse
    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
        target_label=target_label
    )

    write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, args.eval_scores_file)

    repo_name = "bart-text-style-transfer-llrpd"
    username = "shubhamkr"

    # Create the repo on Hugging Face (if not exists)
    create_repo(f"{username}/{repo_name}", exist_ok=True)

    # Save locally first
    save_dir = f"/content/text-style-transfer/{repo_name}"
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    # Then push to Hub
    model.push_to_hub(f"{username}/{repo_name}")
    tokenizer.push_to_hub(f"{username}/{repo_name}")

### Start Training


In [18]:
%cd /content/text-style-transfer/


from types import SimpleNamespace
import os
os.environ["WANDB_DISABLED"] = "true"
os.makedirs("./output/pos_to_neg/", exist_ok=True)


if __name__ == "__main__":
    args = SimpleNamespace(
        seed_value=53,
        model_name="facebook/bart-base",
        max_length=128,
        batch_size=8,
        learning_rate=5e-5,
        weight_decay=0.0,
        num_train_epochs=3,
        dropout=0.0,
        attention_dropout=0.0,
        activation_dropout=0.0,
        label_smoothing_factor=0.0,

        train_file="./data/train.csv",
        dev_file="./data/dev.csv",
        test_file="./data/test.pkl",
        unparallel_data="./data/unparallel.pkl",

        pred_file="./output/pos_to_neg/pred.pkl",
        src_file="./output/pos_to_neg/src.pkl",
        ref_file="./output/pos_to_neg/trg.pkl",
        output_file = "./output/pos_to_neg/pred.pkl",
        synthetic_train_file="./data/synthetic_train.csv",
        eval_scores_file="./output/pos_to_neg/scores.txt",
        prompt_enabled=True,
        src_col="pos",
        trg_col="neg",
        dev_size=0.1,
        train_pkl_path="./output/pos_to_neg/train.pkl",
        task='pos_to_neg'
    )

# main(args)

/content/text-style-transfer


## Evaluate and Verify with Hub Pushed Trained Model Loaded

In [21]:
def check_evaluation(args):
    shutil.rmtree('facebook', ignore_errors=True)
    set_seed(args.seed_value)

    train_df = remove_newline(pd.read_csv(args.train_file))
    train_df = train_df.sample(frac = 1, random_state = args.seed_value)

    dev_df = remove_newline(pd.read_csv(args.dev_file))
    dev_df = dev_df.sample(frac = 1, random_state = args.seed_value)

    test_df = remove_newline(pd.read_pickle(args.test_file))

    neg_prompt = 'NEG: '
    pos_prompt = 'POS: '
    if args.prompt_enabled:

        train_df["pos"] = train_df["pos"].apply(lambda x: pos_prompt+x)
        train_df["neg"] = train_df["neg"].apply(lambda x: neg_prompt+x)

        dev_df["pos"] = dev_df["pos"].apply(lambda x: pos_prompt+x)
        dev_df["neg"] = dev_df["neg"].apply(lambda x: neg_prompt+x)

        test_df["pos"] = test_df["pos"].apply(lambda x: pos_prompt+x)
        test_df["neg"] = test_df["neg"].apply(lambda x: neg_prompt+x)

    with open('./output/pos_to_neg/src.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.src_col].values.tolist(), pos_prompt), f)

    with open('./output/pos_to_neg/trg.pkl', 'wb') as f:
        pickle.dump(remove_prefix(test_df[args.trg_col].values.tolist(), neg_prompt), f)

    model_name = "shubhamkr/bart-tst-llrpd-postoneg"

    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()
    train_dataset = tokenize_datasets(train_df, args.src_col, args.trg_col, tokenizer, args.max_length)
    dev_dataset = tokenize_datasets(dev_df, args.src_col, args.trg_col, tokenizer, args.max_length)

    # Generate predictions on test data
    generate_predictions(model, test_df, args.src_col, tokenizer, args.pred_file) ###
    print('Completed.')

    evaluator = AutomaticEvaluator(args.seed_value)
    evaluator.set_seed()
    evaluator.load_models()

    target_label = 'NEGATIVE' if args.task == 'pos_to_neg' else 'POSITIVE'

    # <-- Call evaluate_all with the new arguments from argparse
    acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl = evaluator.evaluate_all(
        srcs_file=args.src_file,
        refs_file=args.ref_file,
        preds_file=args.pred_file,
        target_label=target_label
    )

    write_evaluation_results(acc, bleu_withsrc, bleu_withtrg, sim, gpt2_ppl, args.eval_scores_file)


In [ ]:
check_evaluation(args)